In [ ]:
%load_ext autoreload
%autoreload 

In [ ]:
import pandas as pd
import numpy as np
from os import path, makedirs
from datetime import datetime

# local imports
import sys
sys.path.append('../../../')
from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *

from makedf.mcstat import get_MCstat_unc

# turn off PerformanceWarning 
# triggered by mismatched column levels
import warnings
warnings.filterwarnings("ignore", category=pd.errors.PerformanceWarning)

In [ ]:
save_result = True
save_fig = save_result

save_fig_base_dir = "/exp/sbnd/data/users/lpelegri/syst/"

today_str = datetime.now().strftime("%Y%m%d")
save_fig_dir = path.join(save_fig_base_dir, "systematics-other-{}".format(today_str))

if save_fig:
    if not path.exists(save_fig_dir):
        makedirs(save_fig_dir)
    print("saving plots in ", save_fig_dir)

# Load df

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

#Load CV dataframe
keys2load = ["cc1pi", "hdr", "histpotdf", "nudf"] ## keys from the configuration file
mc_bnb_df = load_df("/exp/sbnd/data/users/lpelegri/cafpyana_data/cc1pi_extended_syst.df", keys2load, 100)
mc_evt_df = mc_bnb_df['cc1pi']
mc_nu_df = mc_bnb_df['nudf']
mc_hdr_df = mc_bnb_df['hdr']

#Add weight column
data_tot_pot = 5.947e+18
mc_tot_pot = mc_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_evt_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_evt_df))

#Do truth matchign
mc_evt_df = perform_truth_matching(mc_evt_df, mc_nu_df)
mc_nu_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_nu_df))

# Perform selection

In [ ]:
mc_obvious_cosmic_mask = mc_evt_df.slc.cut.obvious_cosmic
mc_t0_mask = mc_evt_df.slc.cut.t0
mc_is_inside_FV_mask = mc_evt_df.slc.cut.inside_FV
mc_nu_score_mask = mc_evt_df.slc.cut.nu_score
mc_track_mask = mc_evt_df.slc.cut.track
mc_shower_mask = mc_evt_df.slc.cut.shower 
mc_chi2_mask = mc_evt_df.slc.cut.MIP_candidates 
mc_angle_mask = mc_evt_df.slc.cut.angle 
mc_proton_BDT_mask = mc_evt_df.slc.cut.proton_BDT
mc_containment_mask = mc_evt_df.slc.cut.containment 
mc_michel_mask = mc_evt_df.slc.cut.michel 
mc_extra_pion_mask = mc_evt_df.slc.cut.extra_pion 
mc_energy_mask = mc_evt_df.slc.cut.energy

# 1. Define the order of cuts
mc_cut_sequence = [
    ("cosmic", mc_obvious_cosmic_mask),
    ("t0", mc_t0_mask),
    ("FV", mc_is_inside_FV_mask),
    ("nu_score", mc_nu_score_mask),
    ("track", mc_track_mask),
    ("chi2", mc_chi2_mask),
    ("shower", mc_shower_mask),
    ("angle", mc_angle_mask),
    ("proton_BDT", mc_proton_BDT_mask),
    ("containment", mc_containment_mask),
    ("michel", mc_michel_mask),
    ("extra_pion", mc_extra_pion_mask),
    ("energy", mc_energy_mask)
]

# 2. Build the cumulative masks
mc_cumulative_mak = None

for name, mask in mc_cut_sequence:
    if mc_cumulative_mak is None:
        mc_cumulative_mak = mask
    else:
        mc_cumulative_mak = mc_cumulative_mak & mask

In [ ]:
print(mc_evt_df.truth.nu_categ.value_counts())

In [ ]:
#mc_evt_df = mc_evt_df[mc_cumulative_mak]

In [ ]:
#make it a slc df
mc_evt_df = (
        mc_evt_df
        .groupby(['__ntuple', 'entry', 'rec.slc..index'])
        .first()
    )
mc_evt_df = mc_evt_df.sort_index()


In [ ]:
print("==== breakdown of selected events ====")
print(mc_evt_df.truth.nu_categ.value_counts())
#print(mc_evt_df.genie_categ.value_counts())

In [ ]:

flux_systematics = [
    'expskin_Flux',
    #'kzero_Flux',
    #'horncurrent_Flux',
    #'kminus_Flux',
    #'kplus_Flux',
    #'nucleoninexsec_Flux',
    #'nucleonqexsec_Flux',
    #'nucleontotxsec_Flux',
    #'piminus_Flux',
    #'pionqexsec_Flux',
    #'pioninexsec_Flux',
    #'piontotxsec_Flux',
    'piplus_Flux'
]

# Flux

In [ ]:
'''
import matplotlib.pyplot as plt
import numpy as np

plot_vs_energy = True   # <-- boolean switch
E_col = ('truth','E','','','','')

for syst in flux_systematics:

    plt.figure()

    for i in range(101):

        col = ('truth', syst, f'univ_{i}', '', '', '')

        if col not in mc_evt_df.columns:
            continue

        y = mc_evt_df[col].values
        mask = np.isfinite(y)

        if plot_vs_energy:

            if E_col not in mc_evt_df.columns:
                raise ValueError("Energy column not found")

            x = mc_evt_df[E_col].values
            mask &= np.isfinite(x)

            plt.scatter(x[mask], y[mask], s=3, alpha=0.3)

        else:

            x = np.full(mask.sum(), i)
            plt.scatter(x, y[mask], s=5)

    if plot_vs_energy:
        plt.xlabel("E")
    else:
        plt.xlabel("Universe")

    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")

    plt.show()
'''

In [ ]:
'''
import numpy as np
import matplotlib.pyplot as plt


for syst in flux_systematics:

    means = []
    stds = []
    medians = []
    universes = []

    for i in range(101):

        col = ('truth', syst, f'univ_{i}', '', '', '')

        if col in mc_evt_df.columns:
            y = mc_evt_df[col].values
            y = y[~np.isnan(y)]
            
            universes.append(i)
            means.append(np.mean(y))
            stds.append(np.std(y))
            medians.append(np.median(y))

    means = np.array(means)
    stds = np.array(stds)
    medians = np.array(medians)
    plt.figure()

    # ±2σ band
    plt.bar(
        universes,
        4*stds,
        bottom=means-2*stds,
        width=0.8,
        alpha=0.2,
        label="±2σ"
    )

    # ±1σ band
    plt.bar(
        universes,
        2*stds,
        bottom=means-stds,
        width=0.5,
        alpha=0.5,
        label="±1σ"
    )

    # mean marker
    plt.scatter(universes, means, marker='o', s=40, label="Mean")

    # median marker
    plt.scatter(universes, medians, marker='x', s=40, label="Median")

    plt.xlabel("Universe")
    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")
    plt.legend()

    plt.show()
'''

# G4

In [ ]:
g4_systematics = [
    #'reinteractions_kminus_Geant4',
    #'reinteractions_kplus_Geant4',
    'reinteractions_neutron_Geant4',
    'reinteractions_piminus_Geant4',
    #'reinteractions_piplus_Geant4',
    'reinteractions_proton_Geant4'
]
label_map = {
    "reinteractions_kminus_Geant4": r"$K^{-}$",
    "reinteractions_kplus_Geant4": r"$K^{+}$",
    "reinteractions_neutron_Geant4": "n",
    "reinteractions_piminus_Geant4": r"$\pi^{-}$",  # Changed # to \
    "reinteractions_piplus_Geant4": r"$\pi^{+}$",   # Changed # to \
    "reinteractions_proton_Geant4": "p",
}

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_vs_energy = False   # <-- boolean switch
E_col = ('truth','E','','','','')
genie_categs = ["other", "cosmic", "out_AV_nu", "nu_mu_NC" , "nu_mu_CC_QE" , "nu_mu_CC_MEC" , "nu_mu_CC_Res", "nu_mu_CC_Dis" ]
for genie_categ in genie_categs:
    genie_df = mc_evt_df[mc_evt_df.truth.genie_categ == genie_categ].copy()
    
    for syst in g4_systematics:
        
        plt.figure()
    
        for i in range(101):
    
            col = ('truth', syst, f'univ_{i}', '', '', '')
    
            if col not in genie_df.columns:
                continue
    
            y = genie_df[col].values
            mask = np.isfinite(y)
    
            if plot_vs_energy:
    
                if E_col not in genie_df.columns:
                    raise ValueError("Energy column not found")
    
                x = genie_df[E_col].values
                mask &= np.isfinite(x)
    
                plt.scatter(x[mask], y[mask], s=3, alpha=0.3)
    
            else:
    
                x = np.full(mask.sum(), i)
                plt.scatter(x, y[mask], s=5)
    
        if plot_vs_energy:
            plt.xlabel("E")
        else:
            plt.xlabel("Universe")
    
        plt.ylabel("Value")
        plt.title(f"Syst: {syst}, genie categ = {genie_categ}")
    
        plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plot_vs_energy = False   # <-- boolean switch
E_col = ('truth','E','','','','')

for genie_categ in 
for syst in g4_systematics:
    
    plt.figure()

    for i in range(101):

        col = ('truth', syst, f'univ_{i}', '', '', '')

        if col not in mc_evt_df.columns:
            continue

        y = mc_evt_df[col].values
        mask = np.isfinite(y)

        if plot_vs_energy:

            if E_col not in mc_evt_df.columns:
                raise ValueError("Energy column not found")

            x = mc_evt_df[E_col].values
            mask &= np.isfinite(x)

            plt.scatter(x[mask], y[mask], s=3, alpha=0.3)

        else:

            x = np.full(mask.sum(), i)
            plt.scatter(x, y[mask], s=5)

    if plot_vs_energy:
        plt.xlabel("E")
    else:
        plt.xlabel("Universe")

    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")

    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

for syst in g4_systematics:

    means = []
    stds = []
    medians = []
    universes = []

    for i in range(101):

        col = ('truth', syst, f'univ_{i}', '', '', '')

        if col in mc_evt_df.columns:
            y = mc_evt_df[col].values
            y = y[~np.isnan(y)]
            
            universes.append(i)
            means.append(np.mean(y))
            stds.append(np.std(y))
            medians.append(np.median(y))

    means = np.array(means)
    stds = np.array(stds)
    medians = np.array(medians)
    plt.figure()

    # ±2σ band
    plt.bar(
        universes,
        4*stds,
        bottom=means-2*stds,
        width=0.8,
        alpha=0.2,
        label="±2σ"
    )

    # ±1σ band
    plt.bar(
        universes,
        2*stds,
        bottom=means-stds,
        width=0.5,
        alpha=0.5,
        label="±1σ"
    )

    # mean marker
    plt.scatter(universes, means, marker='o', s=40, label="Mean")

    # median marker
    plt.scatter(universes, medians, marker='x', s=40, label="Median")

    plt.xlabel("Universe")
    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")
    plt.legend()

    plt.show()

# GENIE

In [ ]:
genie_systematics_multisim = [
    'GENIEReWeight_SBN_v1_multisim_RPA_CCQE',
    'GENIEReWeight_SBN_v1_multisim_CoulombCCQE',
    'GENIEReWeight_SBN_v1_multisim_NormCCMEC',
    'GENIEReWeight_SBN_v1_multisim_NormNCMEC',
    'GENIEReWeight_SBN_v1_multisim_RDecBR1gamma',
    'GENIEReWeight_SBN_v1_multisim_RDecBR1eta',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpCC2pi',
    '#GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvpNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvnNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarpNC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnCC2pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC1pi',
    'GENIEReWeight_SBN_v1_multisim_NonRESBGvbarnNC2pi',
]

genie_systematics_multisigma = [
    "GENIEReWeight_SBN_v1_multisigma_VecFFCCQEshape",
    'GENIEReWeight_SBN_v1_multisigma_ZExpA1CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA2CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA3CCQE',
    'GENIEReWeight_SBN_v1_multisigma_ZExpA4CCQE',
    "GENIEReWeight_SBN_v1_multisigma_DecayAngMEC",
    "GENIEReWeight_SBN_v1_multisigma_Theta_Delta2Npi",
    "GENIEReWeight_SBN_v1_multisigma_ThetaDelta2NRad",
    "GENIEReWeight_SBN_v1_multisigma_MaCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MaNCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvCCRES",
    "GENIEReWeight_SBN_v1_multisigma_MvNCRES",
    'GENIEReWeight_SBN_v1_multisigma_AhtBY',
    'GENIEReWeight_SBN_v1_multisigma_BhtBY',
    'GENIEReWeight_SBN_v1_multisigma_CV1uBY',
    'GENIEReWeight_SBN_v1_multisigma_CV2uBY',
    "GENIEReWeight_SBN_v1_multisigma_NormCCCOH", # Handled by re-tuning
    "GENIEReWeight_SBN_v1_multisigma_NormNCCOH",
    'GENIEReWeight_SBN_v1_multisigma_MFP_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_pi',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_pi',
    'GENIEReWeight_SBN_v1_multisigma_MFP_N',
    'GENIEReWeight_SBN_v1_multisigma_FrCEx_N',
    'GENIEReWeight_SBN_v1_multisigma_FrInel_N',
    'GENIEReWeight_SBN_v1_multisigma_FrAbs_N',
    'GENIEReWeight_SBN_v1_multisigma_FrPiProd_N',
    'GENIEReWeight_SBN_v1_multisigma_MaNCEL',
    'GENIEReWeight_SBN_v1_multisigma_EtaNCEL',
]



In [ ]:
print(mc_evt_df.truth.GENIEReWeight_SBN_v1_multisigma_FrCEx_N.columns)

In [ ]:
for syst in genie_systematics_multisim:

    plt.figure()
    for i in range(101):  # universes 0..100
        col = ('truth', syst, f'univ_{i}', '', '', '')
        if col in mc_evt_df.columns:
            y = mc_evt_df[col].values
            x = [i] * len(y)  # x coordinate = universe index
            plt.scatter(x, y, s=5)

    plt.xlabel("Universe")
    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")
    plt.show()

In [ ]:
import matplotlib.pyplot as plt

shift_cols = ['cv','ps1','ps2','ps3','ms1','ms2','ms3']

for syst in genie_systematics_multisigma:

    plt.figure()

    cols = [('truth', syst, shift, '', '', '') for shift in shift_cols]

    # keep only columns that exist
    cols = [c for c in cols if c in mc_evt_df.columns]

    df_syst = mc_evt_df[cols]

    # plot one line per row
    for _, row in df_syst.iterrows():
        plt.plot(range(len(cols)), row.values, alpha=0.2)

    plt.xticks(range(len(cols)), [c[2] for c in cols])
    plt.xlabel("Shift")
    plt.ylabel("Value")
    plt.title(f"Syst: {syst}")

    plt.show()